# Charlotte Project · Week 11 Class 4
## Industrial formats and feature engineering — interactive teaching notebook

**Duration:** 180 minutes · **Audience:** emerging data/ML practitioners · **Purpose:** Learn transferable feature engineering while meeting the curriculum's LAS/SCADA requirement.

**Learning outcomes:** (1) distinguish file formats from source systems; (2) read CSV/JSON and inspect LAS; (3) create meaningful rolling, cumulative, and differential features; (4) validate joins, units, ordering and missing values; (5) describe S3/Glue/Parquet implications.

**Important:** This notebook is a teaching resource. The independent lab uses separate `.py` code and a different dataset. All sample sensor readings are synthetic. No real engineering decisions should be made from them.

**Setup** `pip install pandas numpy scikit-learn ipykernel` and optionally `pip install lasio` to run the LAS cell. Download the student ZIP and open the notebook in VS Code.

## A. Understanding files, data sources, and schemas (30 min)

**What is a data format?** A format specifies how bytes represent a dataset. A **CSV** is a text table; **JSON** represents nested objects and arrays; **Parquet** stores columnar typed data; **LAS** is an industry-standard representation of depth-indexed well logs. A SCADA system, by contrast, is a supervisory control and data acquisition environment—not a universal file type. SCADA measurements may be exported as CSV, JSON, database rows or streams.

**Why this matters:** Different formats preserve different information. CSV has weak intrinsic type metadata and requires explicit conversion; JSON can nest events; Parquet supports typed columns and compression; LAS includes curve names and units. Before making features, validate data dictionaries, physical units, keys, sort order, null markers and data provenance.

**Common mistakes:** treating a column with numbers stored as strings as numeric; assuming two datasets share time or depth alignment; silently overwriting a missing value; assuming measurement units from a header label. **Try it:** compare the structure of the next two inputs.

In [7]:
from pathlib import Path
import pandas as pd, numpy as np, json
DATA_DIR = Path('..')/'lab'/'data' if (Path('..')/'lab'/'data').exists() else Path('data')
scada = pd.read_csv(DATA_DIR/'scada_synthetic.csv')
print(scada.dtypes)
print(scada.head(3).to_string(index=False))
with open(DATA_DIR/'sample_api_payload.json', encoding='utf-8') as f:
    message=json.load(f)
print('Gateway source:',message['source'])
print('First measurement:',message['measurements'][0])

well_id               str
depth_m             int64
pressure_psi      float64
temperature_c     float64
flow_m3_h         float64
operating_mode        str
dtype: object
well_id  depth_m  pressure_psi  temperature_c  flow_m3_h operating_mode
 WELL-B     1000        2788.0           65.0      12.50     production
 WELL-B     1001        2832.0           66.8      12.95     production
 WELL-B     1002        2876.0           68.6      13.40     production
Gateway source: synthetic sensor gateway
First measurement: {'sensor_id': 'P-1', 'reading': 2812.0, 'unit': 'psi'}


### Checking a schema before transformation

A schema describes field names and their expected types. Validate known keys (`well_id`, `depth_m`) and convert measurement strings with `pd.to_numeric(errors='coerce')`, then inspect whether conversion produced unexpected nulls. A quality report is part of the pipeline, not a final afterthought.

In [8]:
required={'well_id','depth_m','pressure_psi','temperature_c','flow_m3_h'}
assert required.issubset(scada.columns)
for name in ['depth_m','pressure_psi','temperature_c','flow_m3_h']:
    scada[name]=pd.to_numeric(scada[name],errors='coerce')
print(scada.isna().sum())
print('Repeated keys:',scada.duplicated(['well_id','depth_m']).sum())

well_id           0
depth_m           0
pressure_psi      1
temperature_c     0
flow_m3_h         1
operating_mode    0
dtype: int64
Repeated keys: 0


## B. Parsing LAS files, plus general-purpose formats (35 min)

**LAS** stands for Log ASCII Standard. It commonly contains a version block, well metadata, curve descriptions, and a depth-indexed data table. Curve names such as `DEPT`, `GR`, and `RHOB` are specialised; units should always be checked. `lasio.read()` loads LAS and `las.df()` returns a DataFrame indexed by depth.

**Workplace caution:** LAS depth-indexed curves and real-time SCADA timestamp readings are not inherently aligned. The lab uses a deliberately shared synthetic depth identifier. In a real system, coordinate reconciliation requires engineering sign-off.

**Why a dedicated reader?** A general `pd.read_csv()` call may misinterpret metadata sections, LAS null values, curve units or version features. Use a domain-specific reader for reliable ingestion and record any assumptions. An example source of public LAS data is Kansas Geological Survey (linked in the notes).

In [9]:
try:
    import lasio
except ImportError:
    print('Optional LAS reader unavailable. Install with: python -m pip install lasio')
    print('The remaining demonstrations will still execute.')
else:
    example=lasio.read(str(DATA_DIR/'well_b_synthetic.las'))
    print('Curves:', [(x.mnemonic,x.unit) for x in example.curves])
    print(example.df().head(3))

Curves: [('DEPT', 'M'), ('GR', 'API'), ('RHOB', 'G/C3')]
          GR  RHOB
DEPT              
1000.0  45.0  2.44
1001.0  47.0  2.45
1002.0  49.0  2.46


### General-purpose feature engineering: botanical data

Features should represent a useful signal, not just more columns. Feature engineering might include ratios, differences, binned values, and transformations. To avoid building a solely energy-specific skillset, the next demonstration uses public Iris measurements and constructs a flower shape ratio. This derived feature is an illustration, not a proven predictive improvement. Always check whether a feature is meaningful and whether it could leak information unavailable at prediction time.

In [10]:
from sklearn.datasets import load_iris
iris=load_iris(as_frame=True).frame.copy()
iris.columns=['sepal_length_cm','sepal_width_cm','petal_length_cm','petal_width_cm','target']
iris['petal_length_width_ratio']=iris['petal_length_cm']/iris['petal_width_cm']
iris['sepal_area_proxy']=iris['sepal_length_cm']*iris['sepal_width_cm']
print(iris[['petal_length_width_ratio','sepal_area_proxy']].head())

   petal_length_width_ratio  sepal_area_proxy
0                       7.0             17.85
1                       7.0             14.70
2                       6.5             15.04
3                       7.5             14.26
4                       7.0             18.00


## C. Core time-aware and order-aware features (45 min)

**Why ordering matters:** rolling and cumulative features depend on the order of observations. For sensor data, order by well and timestamp; for a well log, order by well and depth. Without reliable ordering, `diff()`, `rolling()` and `cumsum()` can be meaningless.

- **Differential:** `x.diff()` = current reading − previous reading. The first row is undefined (`NaN`).
- **Rolling variance:** dispersion among recent readings; `rolling(window=3, min_periods=3).var(ddof=0)` uses only three readings, and is missing until the window fills.
- **Cumulative sampled values:** `cumsum()` sums observations in a chosen order. It is *not* an integrated physical volume if the values are rates; rate-to-volume requires a time interval and unit conversion.
- **Temperature differential:** can be a change over adjacent readings or the difference between two sensors. Name the feature to show which definition you use.

**Common mistakes:** computing rolling features across different wells, looking into future rows, treating zeros as missing data or missing data as zero without documentation.

In [11]:
demo=pd.DataFrame({'well_id':['A']*5,'step':[1,2,3,4,5],
'pressure_psi':[3000,3020,3010,np.nan,3050],
'temperature_c':[65,66,68,67,69],
'flow_m3_h':[8,9,8.5,np.nan,10]})
demo=demo.sort_values(['well_id','step'])
demo['pressure_delta_psi']=demo.groupby('well_id')['pressure_psi'].diff()
demo['pressure_var_3']=demo.groupby('well_id')['pressure_psi'].transform(lambda x: x.rolling(3,min_periods=3).var(ddof=0))
demo['temperature_delta_c']=demo.groupby('well_id')['temperature_c'].diff()
demo['cumulative_sampled_flow']=demo.groupby('well_id')['flow_m3_h'].transform(lambda x:x.fillna(0).cumsum())
print(demo.to_string(index=False))

well_id  step  pressure_psi  temperature_c  flow_m3_h  pressure_delta_psi  pressure_var_3  temperature_delta_c  cumulative_sampled_flow
      A     1        3000.0             65        8.0                 NaN             NaN                  NaN                      8.0
      A     2        3020.0             66        9.0                20.0             NaN                  1.0                     17.0
      A     3        3010.0             68        8.5               -10.0       66.666667                  2.0                     25.5
      A     4           NaN             67        NaN                 NaN             NaN                 -1.0                     25.5
      A     5        3050.0             69       10.0                 NaN             NaN                  2.0                     35.5


### Active learning check

Predict the output before running the next cell: How should the previous and current readings behave at a well boundary? Does the first pressure delta for each well have a meaningful value? Why might the cumulative sum of recorded flow rates be useful as a software feature but *not* as a true fluid volume estimate?

In [ ]:
two_wells=pd.DataFrame({'well_id':['A','A','B','B'], 'depth_m':[100,101,100,101], 'pressure_psi':[10,15,100,102]})
two_wells=two_wells.sort_values(['well_id','depth_m'])
two_wells['delta']=two_wells.groupby('well_id')['pressure_psi'].diff()
print(two_wells.to_string(index=False))

## D. Joining and validating multiple data sources (35 min)

Real engineering pipelines often join measurements from sources with different granularity. **Before joining:** inspect key uniqueness, sort order, units, join type and expected coverage. `validate='one_to_one'` prevents accidental many-to-many joins, and `indicator=True` exposes mismatches. Check whether missing data is introduced by the merge rather than by sensors themselves.

**Classroom design:** We demonstrate the join using a synthetic depth-indexed LAS-shaped table and CSV data. The assessed lab requires students to use `lasio.read` to load an actual LAS file and then validate the join; code from this cell is deliberately not the finished student solution.

In [ ]:
demo_las=pd.DataFrame({'well_id':['WELL-B']*3,'depth_m':[1000,1001,1002],
'gamma_api':[45,47,49], 'density_g_cm3':[2.44,2.45,2.46]})
demo_scada=scada.loc[scada['depth_m'].isin([1000,1001,1002]),['well_id','depth_m','pressure_psi']]
joined=demo_scada.merge(demo_las,on=['well_id','depth_m'],how='left',validate='one_to_one',indicator=True)
print(joined.to_string(index=False))
print('Merge coverage:',joined['_merge'].value_counts().to_dict())

## E. Data contracts, AWS connections and next steps (35 min)

A **data contract** makes assumptions visible: source, units, expected columns, types, key uniqueness, permissible nulls, sampling intervals, and target outputs. Data-quality checks and logs allow others to reproduce and trust a feature dataset.

**AWS DEA-C01 connection (conceptual, not an AWS hands-on lab):** Amazon S3 stores CSV/JSON/Parquet objects; AWS Glue can catalogue and transform data; Athena can query suitably formatted objects. Compare CSV and Parquet when designing analytics storage, and distinguish an ingestion failure from a validation failure. Official domain reference: https://docs.aws.amazon.com/aws-certification/latest/data-engineer-associate-01/data-engineer-associate-01-domain1.html

**Knowledge check:** (1) What prevents a many-to-many merge? (2) How do you create a rolling variance without future look-ahead? (3) Why must a flow rate be multiplied by duration before interpreting its sum as volume? (4) When is a schema-rich columnar format preferable to CSV? (5) Why is a LAS specialist reader preferable to generic CSV parsing?

**Independent lab:** Write separate `.py` code; use `feature/week11-class04-feature-engineering` branch in your own repo; submit a Pull Request. No teaching notebook is required in the assessed PR.